## Read pkl file and save images. 

In [4]:
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


# ==========================================================
# Input / output
# ==========================================================
file_path = r"C:\Users\domin\Downloads\processed_data_full_03_25_2025.pkl"

out_dir = Path("BLO_Extracted_Images")
raw_dir = out_dir / "Raw_Images"
pert_dir = out_dir / "Perturbation_Images"

raw_dir.mkdir(parents=True, exist_ok=True)
pert_dir.mkdir(parents=True, exist_ok=True)


# ==========================================================
# Helper
# ==========================================================
def save_stack(stack, out_folder, prefix, cmap="gray", symmetric=False):
    stack = np.asarray(stack)
    stack = np.squeeze(stack)

    if stack.ndim == 2:
        stack = stack[np.newaxis, :, :]

    if stack.ndim != 3:
        raise ValueError(f"{prefix}: expected 2D or 3D image stack, got shape {stack.shape}")

    print(f"{prefix}: saving stack with shape {stack.shape}")

    for i in range(stack.shape[0]):
        img = stack[i]

        if symmetric:
            vmax = np.nanmax(np.abs(img))
            if not np.isfinite(vmax) or vmax == 0:
                vmax = 1

            plt.imsave(
                out_folder / f"{prefix}_{i:04d}.png",
                img,
                cmap=cmap,
                vmin=-vmax,
                vmax=vmax,
            )

        else:
            plt.imsave(
                out_folder / f"{prefix}_{i:04d}.png",
                img,
                cmap=cmap,
            )

    print(f"Saved {stack.shape[0]} {prefix} images")


# ==========================================================
# Load PKL
# ==========================================================
with open(file_path, "rb") as file:
    data = pickle.load(file)


# ==========================================================
# Extract arrays
# Each list is [name, actual_data]
# ==========================================================
raw_images = data[0][1]
perturbation_images = data[5][1]
date_start = data[7][1]


print("Raw image stack shape:", np.asarray(raw_images).shape)
print("Perturbation image stack shape:", np.asarray(perturbation_images).shape)
print("Date & start time:", date_start)


# ==========================================================
# Save images
# ==========================================================
save_stack(
    raw_images,
    raw_dir,
    prefix="raw",
    cmap="gray",
    symmetric=False,
)

save_stack(
    perturbation_images,
    pert_dir,
    prefix="perturbation",
    cmap="gray",
    symmetric=True,
)


# ==========================================================
# Save date/start time
# ==========================================================
with open(out_dir / "Date_and_Start_Time.txt", "w") as f:
    f.write(f"Date: {date_start[0]}\n")
    f.write(f"Start time: {date_start[1]}\n")


print("\nDone.")
print("Output folder:", out_dir)

Raw image stack shape: (301, 512, 512)
Perturbation image stack shape: (280, 1000, 1000)
Date & start time: ['03_25_2025', '20:29:27']
raw: saving stack with shape (301, 512, 512)
Saved 301 raw images
perturbation: saving stack with shape (280, 1000, 1000)
Saved 280 perturbation images

Done.
Output folder: BLO_Extracted_Images
